<a href="https://colab.research.google.com/github/d-noe/NLP_DH_PSL_Fall2025/blob/main/code/2_topic_modeling/Hands_on_2_CompareTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic Modeling UN General Debates Speeches

Every year, at the opening of the United Nations General Assembly, world leaders gather in New York for the UN General Debates. Each country's representative takes the stage to share their views on global challenges, priorities, and hopes for the future. These speeches offer a fascinating window into international politics and how global concerns evolve over time... but what are they *really* talking about?

![](https://global.unitednations.entermediadb.net/assets/mediadb/services/module/asset/downloads/preset/Libraries/Production%20Library/23-09-2025-UN-GA80-wideview.jpg/image1170x530cropped.jpg)

While running this notebook, feel free to explore different functionalities of BERTopic and LDA!

In [ ]:
!pip install bertopic gensim pyLDAvis

In [ ]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora
from gensim.models import LdaModel

import pyLDAvis.gensim
# Because of some pyLDAvis depreciated method that would add warnigns to every cell... (re-run if still issues w/ warnings)
import warnings
warnings.filterwarnings('ignore')

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import json
import pandas as pd
import geopandas as gpd
import altair as alt

In [ ]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_PSL_Fall2025/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github

## Load the data

Load the UNGDC:

<details><summary>What kind of data exactly?</summary>

The UNGDC (United Nations General Debate Corpus) gathers speeches from nations representatives during the General Debates: they usually speak about their worldview, challenges, efforts, etc. Is it suited (at least in theory) to directly apply TM on these data? If so, (or if not), which kind of TM algorithm?

<details><summary>Hint?</summary>

Two versions of the datasets are actually stored on the GitHub repository:

- `topic_data/ungdc.csv`: 'raw' dataset
- `topic_data/ungdc_coarse_paragraphs.csv`: coarse pre-processing has already been applied (it could be made better) to chunk speeches into paraggraphs, maybe it could be helpful?

</details>

</details>

In [ ]:
# Load the data
filename = "ungdc_coarse_paragraphs" # TODO?
df = load_csv_from_github(f"data/topic_data/{filename}.csv")
print(f"Loaded dataframe of {len(df)} rows, with columns: {', '.join(df.columns)}.")

# Display 5 random rows
df.sample(5)

Very quick exploration of the data, feel free to extend it!

In [ ]:
def unique_country_per_year(
    df,
):
  years = sorted(df["year"].unique().tolist())
  countries_count = [
      len(df[df["year"]==y]["country_iso"].unique())
      for y in years
  ]
  fig, ax = plt.subplots(figsize=(10,4))
  ax.plot(
      years,
      countries_count,
      color="tab:orange",
      marker='o'
  )
  ax.set_xlabel("Year")
  ax.set_ylabel("# unique countries")
  sns.despine()
  plt.show()
  return

In [ ]:
def count_per_year(
    df,
):
  dummy_variable = "session" # use any column, just to count
  dict_year_count = df[["year", dummy_variable]].groupby("year").count().to_dict()[dummy_variable]

  fig, ax = plt.subplots(figsize=(10,4))
  ax.plot(
      dict_year_count.keys(),
      dict_year_count.values(),
      marker='o'
  )
  ax.set_xlabel("Year")
  ax.set_ylabel("Count")
  sns.despine()
  plt.show()
  return


In [ ]:
count_per_year(df)

In [ ]:
unique_country_per_year(df)

🔮 Felling adventurous? Jump directly to the [real problem](#real_problem) section, and test TM algorithms by your own!

# BERTopic / LDA Comparison

Let us now apply both BERTopic and LDA on the data.

To do so, we'll first reduce the amount of data in the corpus, let's say we are especially interested in what representatives from one particular country talked about:

In [ ]:
focus_country_iso = "" # TODO -> pick a country

df_filtered = df[df["country_iso"]==focus_country_iso]

print(len(df_filtered))

In [ ]:
count_per_year(df_filtered)

In [ ]:
unique_country_per_year(df_filtered) # oof

### BERTopic

In [ ]:
documents = df_filtered["text"].tolist()

print(len(documents))

In [ ]:
# Create your representation model
representation_model = KeyBERTInspired()

# Use the representation model in BERTopic on top of the default pipeline
bertopic_model = BERTopic(
    representation_model=representation_model,
    verbose=True
)

topics, probs = bertopic_model.fit_transform(documents)

In [ ]:
bertopic_model.visualize_topics()

In [ ]:
bertopic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)

In [ ]:
bertopic_model.visualize_hierarchy(top_n_topics=50)

You can explore further the methods and functionalities of `BERTopic` (have a look at the [documentation](https://maartengr.github.io/BERTopic/index.html)).

For instance, maybe you think that the number of topics might be a bit too large to obtain a useful representation of the data?
- Try [reducing the number of topics](https://maartengr.github.io/BERTopic/getting_started/topicreduction/topicreduction.html).

### LDA

Ok, now let's apply LDA!

There exists differnet implementation of the algorithm, for instance using [`sklearn`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) (which can be convenient for modularity and further integration), but for now let's stick to one of the most popular one: [`gensim`'s LDA](https://radimrehurek.com/gensim/models/ldamodel.html).

In [ ]:
documents = df_filtered["text"].tolist()

In [ ]:
%%time

# quick pre-processing, could be made more extensive
texts = [
    [word for word in simple_preprocess(doc) if word not in STOPWORDS]
    for doc in documents
]

In [ ]:
%%time

dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

In [ ]:
%%time

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=62, # we've already run BERTopic, let's try with the number of topics it found..
    random_state=42, # LDA is not deterministic!
    passes=10
)

In [ ]:
lda_model.get_topics().shape # N topics x Vocab size --> Topic = distribution over vocab

In [ ]:
for idx, topic in lda_model.print_topics(num_words=5, num_topics=-1): # -1 --> all topics (ordered by significance)
    print(f"Topic #{idx}: {topic}")

In [ ]:
# Visualisation
pyLDAvis.enable_notebook()
pyLDAvis.gensim.prepare(lda_model, corpus, dictionary)


You can further explore the functionalities of `gensim`'s LDA further, for instance why not try to explore the topic distribution of some documents?

## Compare both

Try to explore the topics produced both by BERTopic and LDA.

For instance get the topic representations side by side, retrieve the most representative documents from each topic, etc.

 --> Which one do you prefer? How could you improve the representations of one or the other method?

# Adding additional variables

Let's spice things up a little, let's take into considerations other variables.

For instance we can use:
- `year`: to see how topics discussed evolved across time from 1946 to today
- `country_iso`: compute how distinctive topics can be for different nations.

In the following section, we'll focus on BERTopic model (but same analyses could be lead with LDA, maybe sometimes with a bit additional work, you can try it afterwards!).

## Dynamic topic modeling

Dynamic topic modeling (DTM) is a collection of techniques aimed at analyzing the evolution of topics over time. These methods allow you to understand how a topic is represented across different times.
One of the main goal of TDM is to keep topics constant over time even if the way it is discussed changed (for instance, people may talk differently about environmental awareness than those in 2015; Although the topic itself remains the same (arguably).

Let's apply in on our UN General Debates speeches from USA representatives: how did the themes addressed evolved along time?

Conviniently, `BERTopic` already has a method implemented to perform DTM (`topics_over_time`), you just have to provide the temporal data together with the texts. [See documentation here.](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)

In [ ]:
df = load_csv_from_github("data/topic_data/ungdc_coarse_paragraphs.csv")
df_filtered = df[df["country_iso"]=="USA"]

In [ ]:
# Create your representation model
representation_model = KeyBERTInspired()

# Use the representation model in BERTopic on top of the default pipeline
bertopic_model = BERTopic(
    representation_model=representation_model,
    verbose=True
)

# First we fit the BERTopic
topics, probs = bertopic_model.fit_transform(df_filtered["text"].tolist())

# And then we perform DTM
topics_over_time = bertopic_model.topics_over_time(
    df_filtered["text"].tolist(),
    df_filtered["year"].tolist()
)

In [ ]:
# Now we can directly visualize the trends!

bertopic_model.visualize_topics_over_time(topics_over_time)

## Geographic distribution

Let us now consider the spatial dimension: what are the themes addressed by the representantives of different countries?

First: a visual exploration, we'll visualize the distribution of topics on a map.

Then: statistical analysis: we'll try to find what topics are the most characteristics of different countries through the computation of Pointwise Mutual Information (MPI).

In [ ]:
# To do so let's use the data from all countries between 2010-2020
# If you don't have GPU, reduce to one single year!

df_filtered = df[[y in np.arange(2010, 2020) for y in df["year"]]]
documents = df_filtered["text"].tolist()

print(len(documents))

In [ ]:
count_per_year(df_filtered)

In [ ]:
unique_country_per_year(df_filtered)

### Spatial Exploration

In [ ]:
%%time

#representation_model = KeyBERTInspired()
topic_model = BERTopic(
#    representation_model=representation_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents)

In [ ]:
topic_model.visualize_topics()

In [ ]:
# make a document <-> topic dataframe

df_topics = pd.DataFrame()
df_topics["text"] = documents
df_topics["topic"] = topics
df_topics["prob"] = probs
df_topics["country_iso"] = df_filtered["country_iso"].tolist()

# --- Filter out topic -1: noise
df_topics = df_topics[df_topics["topic"] != -1]

topics_dict = topic_model.get_topics()
topics_dict = {
    k: ', '.join([w[0] for w in v[:5]])
    for k,v in topics_dict.items()
}
df_topics["topic"] = df_topics["topic"].map(topics_dict)

df_topics

In [ ]:
def topic_world_map_topic_share(
    df,
    iso_col="country_iso",
    topic_col="topic",
    width=900,
    height=500,
    title="Share of speeches about a topic by country"
):
    """
    Interactive world choropleth (Altair) showing each country's share of all speeches on a given topic.
    Normalization is across countries per topic (so values sum to 1 for each topic).
    """

    # 1) compute counts per (country, topic)
    counts = df.groupby([iso_col, topic_col]).size().reset_index(name="count")

    # 2) normalize ACROSS countries (for each topic)
    prop = (
        counts.groupby(topic_col)
        .apply(lambda g: g.assign(proportion=g["count"] / g["count"].sum()))
        .reset_index(drop=True)
    )

    # 3) load Natural Earth GeoJSON
    world = gpd.read_file(
        "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
    )

    # 4) merge so each row has geometry + (topic, proportion, count)
    merged = world.merge(prop, left_on="ADM0_A3", right_on=iso_col, how="left")

    # 5) convert to feature list for Altair
    geojson = json.loads(merged.to_json())
    features = geojson["features"]

    # 6) prepare dropdown parameter for topics
    topics = sorted(df[topic_col].astype(str).unique().tolist())
    if not topics:
        raise ValueError("No topics available in the dataframe.")

    topic_param = alt.param(
        name="topic_select",
        value=topics[0],
        bind=alt.binding_select(options=topics, name="Select topic: ")
    )

    # 7) build background layer
    background = (
        alt.Chart(alt.Data(values=features))
        .mark_geoshape(fill="lightgray", stroke="white", strokeWidth=0.3)
        .project("naturalEarth1")
    )

    # 8) expression for filtering selected topic
    expr = f"datum.properties.{topic_col} == topic_select"

    # 9) main map
    map_chart = (
        alt.Chart(alt.Data(values=features))
        .mark_geoshape(stroke="white", strokeWidth=0.3)
        .encode(
            color=alt.Color(
                "properties.proportion:Q",
                title="Share of speeches on topic",
                scale=alt.Scale(scheme="blues"),  # no fixed domain; dynamic scale
            ),
            tooltip=[
                alt.Tooltip("properties.NAME:N", title="Country"),
                alt.Tooltip(f"properties.{topic_col}:N", title="Topic"),
                alt.Tooltip("properties.proportion:Q", title="Share", format=".2%"),
                alt.Tooltip("properties.count:Q", title="Count"),
            ],
        )
        .transform_filter(expr)
        .project("naturalEarth1")
        .properties(width=width, height=height, title=title)
    )

    chart = (background + map_chart).add_params(topic_param)
    return chart

In [ ]:
# This and rendering it can take 3-4 minutes...
chart = topic_world_map_topic_share(df_topics)

In [ ]:
chart

### PMI

Pointwise Mutual Information (PMI) is a classic measure from information theory that quantifies how strongly two events — here, a country and a topic — are associated compared to what we would expect if they were independent. In simple terms, PMI tells us how much more often a country talks about a topic than we'd predict by chance.

When used with topic modeling, PMI provides an interpretable way to identify which topics are particularly distinctive or characteristic of certain countries. While topic proportions show what each country discusses, PMI highlights the surprising associations — the themes that truly set one country apart from the global conversation. This makes it a powerful complement to standard topic modeling analyses, revealing meaningful political or cultural patterns that raw frequencies alone might miss.

[See Wikipedia for more information](https://en.wikipedia.org/wiki/Pointwise_mutual_information)

In [ ]:
def compute_country_topic_pmi(df, iso_col="country_iso", topic_col="topic"):
    """
    Compute Pointwise Mutual Information (PMI) between countries and topics.
    Returns a DataFrame with columns: [country, topic, PMI]
    """
    # Count joint and marginal frequencies
    joint = df.groupby([iso_col, topic_col]).size().reset_index(name="n_ct")
    n_total = len(df)
    joint["p_ct"] = joint["n_ct"] / n_total

    p_c = df[iso_col].value_counts() / n_total
    p_t = df[topic_col].value_counts() / n_total

    # Compute PMI = log2( P(c,t) / (P(c)*P(t)) )
    joint["PMI"] = np.log2(joint["p_ct"] / (joint[iso_col].map(p_c) * joint[topic_col].map(p_t)))

    # Return full DataFrame
    return joint[[iso_col, topic_col, "PMI"]]

In [ ]:
df_pmi = compute_country_topic_pmi(df_topics)
df_pmi.sort_values(by="PMI", ascending=False)

In [ ]:
for country in [
    "FRA", "ITA", "USA", "KEN", "CHN",
]:
  print(f"============================ {country} ============================")
  # sum is dummy (only one value) -> get for one country -> sort values -> print highest PMI
  print(df_pmi.groupby(["country_iso", "topic"]).sum().xs(country, level=0).sort_values("PMI", ascending=False)[:5])

<a name="real_problem"></a>
# 🌍 Real data, real problems

Most often, the exploration of real-world large corpora through topic modeling methods requires several steps, back and forth (trying to steer the parameters in a good direction), to be able to extract meaningful and interesting topics.

For such unsupervised, hard to evaluate, tasks, the best judge remains yourself: Explore topic modeling techniques on the whole corpus of UN General Debates speeches transcripts (or a sample if you don't have access to enough computational resources).

Begin by picking the model of your choice and dig into the data!

It is likely that the first shot would yield disappointing results (do not panick, ). Try to think systematically, and learn from the outputs. What would you expect? How could you improve it?

- Are the assumptions of the model compatible with my documents?
- Are there any pre-processing steps that could potentially help the model?
- What parameters can be tuned to enhance topic representations (in what directions)?
- Are there different modules that could be plugged? Would they be better suited for the use-case?